In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import iirnotch, filtfilt, find_peaks

In [ ]:
#Part 1: read the raw counts 
daq = SimulatedDAQ(seed=42)
print("Board is on.")

# 1. When the heart is resting, roughly what count do you expect?
baseline_guess = 512    # middle of 0 to 1023, i.e. about 2.5 V
# 2. Roughly how many beats should we see in 2 seconds?
beats_guess = 2         # 60 to 100 bpm gives about 2 to 3 beats in 2 s

check_part1_predictions(baseline_guess, beats_guess)

daq.read_adc(channel=0, n_samples=5, fs=250)

fs = 250            # samples per second
n_samples = 500     # how many samples to read

counts = daq.read_adc(channel=0, n_samples=n_samples, fs=fs)

plt.figure(figsize=(9, 3))
plt.plot(counts)
plt.xlabel("sample number")
plt.ylabel("ADC counts")
plt.show()

In [ ]:
#Part 2: The time axis

expected_spacing = 0.004
expected_last_time = 499 * 0.004

check_time_predictions(expected_spacing, expected_last_time)

t = np.arange(n_samples) / fs

check_time_axis(t, expected_spacing, expected_last_time)

plt.figure(figsize=(9, 3))
plt.plot(t, counts)
plt.xlabel("time (s)")
plt.ylabel("ADC counts")
plt.show()

In [ ]:
#Part 3: Counts to volts 
def counts_to_volts(counts):
    return counts * 5.0 / 1024

volts = counts_to_volts(counts)
print(volts[:5])

In [ ]:
#Part 4: Plotting the voltage signal
plt.figure(figsize=(9, 3))
plt.plot(t, volts)
plt.xlabel("time (s)")
plt.ylabel("voltage (V)")
plt.show()

In [ ]:
#Part 5: Remove 50Hz hum 
def remove_hum(volts, fs):
    b, a = iirnotch(w0=50, Q=30, fs=fs)
    return filtfilt(b, a, volts)

check_hum_removal(remove_hum, volts, fs)

clean = remove_hum(volts, fs)

plt.figure(figsize=(9, 3))
plt.plot(t, volts, alpha=0.5, label="before")
plt.plot(t, clean, label="after")
plt.xlabel("time (s)")
plt.ylabel("voltage (V)")
plt.legend()
plt.show()

In [ ]:
#Part 6: find the heart rate 
def find_beats(signal, fs):
    baseline = np.median(signal)                 # the flat part
    peak_height = np.max(signal) - baseline      # R wave height above the flat part
    peaks, _ = find_peaks(
        signal,
        height=baseline + 0.6 * peak_height,     # 60% of the way up from the flat part
        distance=int(0.4 * fs),                  # peaks at least 0.4 s apart
    )
    return peaks

beats = find_beats(clean, fs)
show_beats(t, clean, beats)

peak_times = beats / fs            # sample numbers -> seconds
gaps = np.diff(peak_times)         # seconds between neighbouring beats
print("gaps (s):", gaps)

bpm = 60 / np.mean(gaps)
print(f"heart rate: {bpm:.0f} bpm")